# SynthDet — Active Learning Demo

This notebook demonstrates the **active learning pipeline**: iteratively generate targeted synthetic data, train a YOLO model, evaluate weaknesses, and refine the generation strategy based on model feedback.

**Active Learning Loop (per iteration):**
1. **Generate** — synthesis strategy targets dataset gaps (exploration) and model weaknesses (exploitation)
2. **Merge** — combine original + all synthetic data (cumulative)
3. **Train** — warm-start YOLO from previous iteration's best weights
4. **Evaluate** — per-bucket/region recall breakdown → `ModelPerformanceProfile`
5. **Refine** — feed profile back into strategy generator for next iteration

The loop stops at max iterations or when mAP improvement converges (< threshold for 2 consecutive iterations).

**Dataset:** Laptop Defects All Sides (same as `demo_pipeline.ipynb`)

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
import sys, os, gc, random, warnings
from pathlib import Path
import os                                                                                                                                                     
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # use GPU 1 (second GPU)
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def resize_if_needed(img, max_size):
    h, w = img.shape[:2]
    if max(h, w) <= max_size:
        return img
    scale = max_size / max(h, w)
    return cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)


warnings.filterwarnings("ignore", category=UserWarning)
%matplotlib inline

PROJECT_ROOT = './laptop defects all sides.v1-v1-raw.yolov12/'
sys.path.insert(0, '../')

#from synthdet.utils.image import resize_if_needed

# ── Paths ────────────────────────────────────────────────────────────────
DATA_YAML = PROJECT_ROOT + "data.yaml"
OUTPUT_DIR = Path("active_learning_output")
OUTPUT_DIR.mkdir(exist_ok=True)

MAX_IMAGE_SIZE = 1280
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Visualization helpers ────────────────────────────────────────────────
COLORS = [
    "#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6",
    "#1abc9c", "#e67e22", "#2980b9", "#c0392b", "#27ae60",
]

BUCKET_ORDER = ["tiny", "small", "medium", "large"]
BUCKET_COLORS = {"tiny": "#e74c3c", "small": "#f39c12", "medium": "#3498db", "large": "#2ecc71"}
REGION_ORDER = [
    "top_left", "top_center", "top_right",
    "middle_left", "middle_center", "middle_right",
    "bottom_left", "bottom_center", "bottom_right",
]


def show_bboxes(img_rgb, bboxes, class_names=None, ax=None, title=None):
    """Draw bboxes with class labels on an image."""
    if ax is None:
        _, ax = plt.subplots()
    h, w = img_rgb.shape[:2]
    ax.imshow(img_rgb)
    for b in bboxes:
        x1 = (b.x_center - b.width / 2) * w
        y1 = (b.y_center - b.height / 2) * h
        bw, bh = b.width * w, b.height * h
        color = COLORS[b.class_id % len(COLORS)]
        rect = mpatches.FancyBboxPatch(
            (x1, y1), bw, bh, linewidth=2,
            edgecolor=color, facecolor="none",
            boxstyle="round,pad=0",
        )
        ax.add_patch(rect)
        if class_names and b.class_id < len(class_names):
            label = class_names[b.class_id]
        else:
            label = str(b.class_id)
        ax.text(
            x1, y1 - 4, label, fontsize=6, color="white",
            bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor="none"),
        )
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)


def show_preds(img_rgb, pred_tuples, class_names, ax, title):
    """Draw predicted bboxes with confidence scores."""
    h, w = img_rgb.shape[:2]
    ax.imshow(img_rgb)
    for bbox, conf in pred_tuples:
        x1 = (bbox.x_center - bbox.width / 2) * w
        y1 = (bbox.y_center - bbox.height / 2) * h
        bw, bh = bbox.width * w, bbox.height * h
        color = COLORS[bbox.class_id % len(COLORS)]
        rect = mpatches.FancyBboxPatch(
            (x1, y1), bw, bh, linewidth=2,
            edgecolor=color, facecolor="none",
            boxstyle="round,pad=0",
        )
        ax.add_patch(rect)
        label = class_names[bbox.class_id] if bbox.class_id < len(class_names) else str(bbox.class_id)
        ax.text(
            x1, y1 - 4, f"{label} {conf:.0%}", fontsize=6, color="white",
            bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor="none"),
        )
    ax.axis("off")
    ax.set_title(title, fontsize=10)


def to_rgb(bgr):
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


print(f"Dataset:   {DATA_YAML}")
print(f"Output:    {OUTPUT_DIR.resolve()}")
print(f"Image cap: {MAX_IMAGE_SIZE}px")
print("Setup complete.")

In [ ]:
import torch
print(torch.cuda.current_device())       # should print 0 (the only "visible" device)
print(torch.cuda.get_device_name(0))  

---
## Step 1: Load & Analyze Dataset

Parse the YOLO dataset and compute statistics — class distribution, size buckets, spatial coverage.

In [ ]:
from synthdet.analysis.loader import load_yolo_dataset
from synthdet.analysis.statistics import compute_dataset_statistics
from synthdet.config import SynthDetConfig
from synthdet.types import BBoxSizeBucket, SpatialRegion

dataset = load_yolo_dataset(DATA_YAML)
base_config = SynthDetConfig(max_image_size=MAX_IMAGE_SIZE)
stats = compute_dataset_statistics(dataset, base_config.analysis)

print(f"Classes ({len(dataset.class_names)}): {dataset.class_names}")
print(f"Train: {len(dataset.train)} | Valid: {len(dataset.valid)} | Test: {len(dataset.test)}")
print(f"Total annotations: {stats.total_annotations}")
print(f"Negative images:   {sum(1 for r in dataset.train if r.is_negative)}")
print(f"Bucket uniformity: {stats.bucket_uniformity:.3f}")
print(f"Spatial uniformity: {stats.region_uniformity:.3f}")

# ── 3-panel overview ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: annotations per class
class_counts = {dataset.class_names[cd.class_id]: cd.count for cd in stats.class_distributions}
names = sorted(class_counts, key=class_counts.get)
counts = [class_counts[n] for n in names]
colors = [COLORS[dataset.class_names.index(n) % len(COLORS)] for n in names]
bars = axes[0].barh(names, counts, color=colors, edgecolor="white")
for bar, c in zip(bars, counts):
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                 str(c), va="center", fontsize=9)
axes[0].set_title("Annotations per Class", fontweight="bold")
axes[0].set_xlabel("Count")

# Panel 2: size bucket distribution (%)
total_ann = max(stats.total_annotations, 1)
bucket_pcts = [stats.overall_bucket_counts.get(BBoxSizeBucket(b), 0) / total_ann * 100
               for b in BUCKET_ORDER]
bars2 = axes[1].barh(BUCKET_ORDER, bucket_pcts,
                      color=[BUCKET_COLORS[b] for b in BUCKET_ORDER], edgecolor="white")
for bar, v in zip(bars2, bucket_pcts):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                 f"{v:.1f}%", va="center", fontsize=9)
axes[1].set_title("Size Bucket Distribution", fontweight="bold")
axes[1].set_xlabel("%")
axes[1].invert_yaxis()

# Panel 3: spatial heatmap
grid = np.zeros((3, 3))
for i, reg in enumerate(REGION_ORDER):
    r, c = divmod(i, 3)
    grid[r, c] = stats.overall_region_counts.get(SpatialRegion(reg), 0)
im = axes[2].imshow(grid, cmap="YlOrRd")
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, str(int(grid[i, j])), ha="center", va="center",
                     fontsize=12, fontweight="bold")
axes[2].set_xticks([0, 1, 2]); axes[2].set_xticklabels(["Left", "Center", "Right"])
axes[2].set_yticks([0, 1, 2]); axes[2].set_yticklabels(["Top", "Middle", "Bottom"])
axes[2].set_title("Spatial Distribution", fontweight="bold")
plt.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

---
## Step 2: Configure Active Learning

Three configs control the loop:

- **`PipelineConfig`** — generation methods, prompts, split ratios
- **`TrainingConfig`** — YOLO architecture, epochs, batch size, early stopping
- **`ActiveLearningConfig`** — number of iterations, convergence threshold, data accumulation mode

The loop is **method-aware**: it selects generation methods based on available API credentials, just like the standard pipeline.

In [ ]:
from synthdet.config import (
    SynthDetConfig,
    AnalysisConfig,
    CompositorConfig,
    InpaintingConfig,
    #ModifyAnnotateConfig,
    TrainingConfig,
    ActiveLearningConfig,
)
from synthdet.pipeline.config_schema import PipelineConfig

# ── Active Learning Parameters ────────────────────────────────────────
N_ITERATIONS = 30       # number of generate-train-evaluate cycles
TRAIN_EPOCHS = 25      # epochs per iteration (patience will early-stop)
CONVERGENCE_THRESH = 0.005  # stop if mAP50 improves < this for 2 iters

# ── Custom prompts (same as demo_pipeline) ────────────────────────────
INPAINTING_PROMPTS = {
    "hard scratch": [
        "A deep scratch gouged into a metal laptop surface, with visible groove depth",
        "Multiple deep parallel scratches on brushed aluminum, harsh lighting",
    ],
    "minor scratch": [
        "A light surface scratch on brushed metal, barely catching the light",
        "A thin cosmetic scratch on a dark matte laptop surface",
    ],
    "sticker marks": [
        "Adhesive residue marks on a smooth laptop surface, slightly cloudy and irregular",
        "Sticker removal residue with partial paper remnants on a flat laptop lid",
    ],
    "broken": [
        "A cracked section of laptop lid",
    ],
    "minor dent": [
        "A small dent on a flat laptop surface, with slight shadow",
    ],
    "minor crack": [
        "A thin hairline crack on a laptop surface, barely visible but distinct",
    ],
}

MODIFY_PROMPTS = {
    "hard scratch": [
        "Add deep, clearly visible white scratch lines gouged into this laptop surface",
    ],
    "minor scratch": [
        "Add clearly visible light scratch lines on this laptop top cover surface",
    ],
    "sticker marks": [
        "Add a clearly visible rectangular patch of sticky adhesive residue on this laptop",
    ],
    "broken": [
        "Add broken area with on this laptop lid",
    ],
    "minor dent": [
        "Add a clearly visible shallow dent creating a noticeable shadow on this laptop",
    ],
    "minor crack": [
        "Add a clearly visible thin dark crack line running across this laptop surface",
    ],
}

INPAINTING_TEMPLATE = (
    "{prompt}. The {class_name} should appear directly on "
    "the laptop lid, matching the existing material and lighting. "
    "Keep the rest of the image unchanged. Photorealistic, top-down view."
)

# ── Select methods based on available credentials ────────────────────
has_api_key = bool(os.environ.get("GOOGLE_API_KEY"))
has_vertex = bool(os.environ.get("GOOGLE_CLOUD_PROJECT"))

if has_vertex:
    run_methods = ["compositor", "inpainting", "generative_compositor"]
    print("GOOGLE_CLOUD_PROJECT detected — running all 4 methods")
elif has_api_key:
    run_methods = ["compositor", "generative_compositor"]
    print("GOOGLE_API_KEY detected — running compositor + generative compositor")
else:
    run_methods = ["compositor"]
    print("No API credentials — running compositor only (offline)")

run_methods = ["compositor", "inpainting", "generative_compositor"]
# ── Build configs ─────────────────────────────────────────────────────
synthdet_cfg = SynthDetConfig(
    max_image_size=MAX_IMAGE_SIZE,
    analysis=AnalysisConfig(
        multiplier=1.0,
        negative_ratio=0.15,
    ),
    compositor=CompositorConfig(
        blend_mode="mixed",
        max_defects_per_image=10,
        rotation_jitter=15.0,
        scale_jitter=(0.7, 1.3),
        valid_zone_margin=0.05,
    ),
    inpainting=InpaintingConfig(
        provider="imagen",
        class_prompts=INPAINTING_PROMPTS,
        prompt_template=INPAINTING_TEMPLATE,
        max_defects_per_image=3,
        max_cost_usd=10.0,
    ),
)

pipeline_cfg = PipelineConfig(
    synthdet=synthdet_cfg,
    methods=run_methods,
    train_split_ratio=0.85,
    validate_output=True,
    augment=False,
)

training_cfg = TrainingConfig(
    model_arch="yolov8n.pt",
    epochs=TRAIN_EPOCHS,
    batch_size=8,
    imgsz=MAX_IMAGE_SIZE,
    patience=10,
    workers=2,
    device="auto",
    project=str(OUTPUT_DIR / "runs"),
    name="active_learning",
    pretrained=True,
)

al_cfg = ActiveLearningConfig(
    max_iterations=N_ITERATIONS,
    convergence_threshold=CONVERGENCE_THRESH,
    accumulate_data=True,       # each iteration builds on all previous synthetic data
    iou_threshold=0.5,
    confidence_threshold=0.25,
    enable_quality_monitoring=False,
    save_intermediate_models=True,
)

# ── Summary ───────────────────────────────────────────────────────────
print(f"\nActive Learning Configuration")
print("=" * 50)
print(f"Iterations:          {N_ITERATIONS}")
print(f"Epochs per iter:     {TRAIN_EPOCHS} (patience={training_cfg.patience})")
print(f"Convergence thresh:  {CONVERGENCE_THRESH}")
print(f"Accumulate data:     {al_cfg.accumulate_data}")
print(f"Methods:             {run_methods}")
print(f"Multiplier:          {synthdet_cfg.analysis.multiplier}x")
print(f"Image size:          {MAX_IMAGE_SIZE}px")

---
## Step 3: Initial Synthesis Strategy (Exploration Only)

Before any training, the strategy is purely **exploration-driven**: fill distribution gaps in size buckets, spatial regions, and class balance. There is no model feedback yet.

In [ ]:
from synthdet.analysis.strategy import generate_synthesis_strategy

initial_strategy = generate_synthesis_strategy(dataset, stats, synthdet_cfg.analysis)

print(f"Target dataset size: {initial_strategy.target_total_images} images")
print(f"Images to generate:  {initial_strategy.total_synthetic_images}")
print(f"Generation tasks:    {len(initial_strategy.generation_tasks)}")
print(f"Has model feedback:  {initial_strategy.has_model_feedback}")
print()

# ── Visualize tasks by priority ──────────────────────────────────────
tasks = initial_strategy.generation_tasks
if tasks:
    fig, ax = plt.subplots(figsize=(12, max(3, len(tasks) * 0.5)))

    rationales = [t.rationale[:55] + "..." if len(t.rationale) > 55 else t.rationale
                  for t in tasks]
    img_counts = [t.num_images for t in tasks]
    priorities = [t.priority for t in tasks]
    bar_colors = plt.cm.RdYlGn(priorities)

    ax.barh(range(len(tasks)), img_counts, color=bar_colors, edgecolor="white")
    ax.set_yticks(range(len(tasks)))
    ax.set_yticklabels(rationales, fontsize=8)
    for i, (cnt, pri) in enumerate(zip(img_counts, priorities)):
        ax.text(cnt + 1, i, f"{cnt} imgs (p={pri:.1f})", va="center", fontsize=8)
    ax.set_title("Initial Strategy: Exploration Tasks (no model feedback yet)", fontweight="bold")
    ax.set_xlabel("Images to generate")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

---
## Step 4: Train Baseline Model

Before running the active learning loop, train a **baseline model** on the original dataset only. This gives us a reference point to measure improvement against.

In [ ]:
from synthdet.training.trainer import YOLOTrainer

print("=" * 60)
print("  Training BASELINE model (original data only)")
print("=" * 60)

baseline_cfg = TrainingConfig(
    model_arch="yolov8n.pt",
    epochs=TRAIN_EPOCHS,
    batch_size=8,
    imgsz=MAX_IMAGE_SIZE,
    patience=10,
    workers=4,
    device="auto",
    project=str(OUTPUT_DIR / "runs"),
    name="baseline",
    pretrained=True,
)

baseline_trainer = YOLOTrainer(baseline_cfg)
baseline_result = baseline_trainer.train(data_yaml=DATA_YAML)

print(f"\nBaseline complete: {baseline_result.epochs_completed} epochs")
print(f"  mAP50:     {baseline_result.best_map50:.4f}")
print(f"  mAP50-95:  {baseline_result.best_map50_95:.4f}")
print(f"  Time:      {baseline_result.training_time_seconds:.0f}s")

gc.collect()

---
## Step 5: Run Active Learning Loop

The `ActiveLearningLoop` orchestrates the full cycle. Each iteration:

1. Generates synthetic data using the current strategy (exploration + exploitation)
2. Merges all data cumulatively (original + iter_0 + iter_1 + ...)
3. Trains YOLO, warm-starting from the previous iteration's best weights
4. Evaluates the model to produce a `ModelPerformanceProfile` (per-bucket/region recall)
5. Feeds the profile back to the strategy generator for the next iteration

The loop stops early if mAP improvement falls below the convergence threshold for 2 consecutive iterations.

In [ ]:
from synthdet.training.loop import ActiveLearningLoop
import logging
logging.basicConfig(level=logging.INFO, force=True)
loop = ActiveLearningLoop(
    data_yaml=DATA_YAML,
    output_dir=OUTPUT_DIR / "al_run",
    pipeline_config=pipeline_cfg,
    training_config=training_cfg,
    al_config=al_cfg,
    seed=SEED,
)

print(f"Starting active learning: {N_ITERATIONS} iterations")
print("=" * 60)
al_result = loop.run()
print("=" * 60)

# ── Summary ───────────────────────────────────────────────────────────
print(f"\nActive Learning Result")
print("=" * 50)
print(f"Iterations completed: {len(al_result.iterations)}")
print(f"Stopped reason:       {al_result.stopped_reason}")
print(f"Final mAP50:          {al_result.final_map50:.4f}")
print(f"Total training time:  {al_result.total_training_time_seconds:.0f}s")
print(f"Total API cost:       ${al_result.total_cost_usd:.2f}")
print(f"Final weights:        {al_result.final_weights}")
print()
print("Per-iteration breakdown:")
print(f"  {'Iter':>4}  {'Records':>8}  {'mAP50':>8}  {'Improve':>8}  {'Cost':>8}  {'Time':>6}")
print(f"  {'----':>4}  {'-------':>8}  {'-----':>8}  {'-------':>8}  {'----':>8}  {'----':>6}")
for ir in al_result.iterations:
    print(
        f"  {ir.iteration + 1:>4}"
        f"  {ir.pipeline_result.total_records:>8}"
        f"  {ir.map50:>8.4f}"
        f"  {'+' if ir.map50_improvement >= 0 else ''}{ir.map50_improvement:>7.4f}"
        f"  ${ir.pipeline_result.total_cost_usd:>7.2f}"
        f"  {ir.training_result.training_time_seconds:>5.0f}s"
    )

gc.collect()

---
## Step 6: mAP Progress Over Iterations

Visualize how detection performance evolves across active learning iterations, compared to the baseline trained on original data only.

In [ ]:
iters = al_result.iterations
iter_nums = [ir.iteration + 1 for ir in iters]
map50s = [ir.map50 for ir in iters]
improvements = [ir.map50_improvement for ir in iters]
records_per_iter = [ir.pipeline_result.total_records for ir in iters]
costs_per_iter = [ir.pipeline_result.total_cost_usd for ir in iters]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: mAP50 progression with baseline reference
axes[0].plot(iter_nums, map50s, "o-", color="#2ecc71", lw=2, markersize=8, label="Active Learning")
axes[0].axhline(baseline_result.best_map50, color="#95a5a6", ls="--", lw=2, label="Baseline")
for x, y in zip(iter_nums, map50s):
    axes[0].text(x, y + 0.01, f"{y:.3f}", ha="center", fontsize=9)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("mAP50")
axes[0].set_title("mAP50 Over Iterations", fontweight="bold")
axes[0].set_xticks(iter_nums)
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel 2: Per-iteration improvement
bar_colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in improvements]
axes[1].bar(iter_nums, improvements, color=bar_colors, edgecolor="white")
axes[1].axhline(CONVERGENCE_THRESH, color="#f39c12", ls="--", lw=1, label=f"Threshold ({CONVERGENCE_THRESH})")
for x, v in zip(iter_nums, improvements):
    axes[1].text(x, v + 0.002, f"{v:+.4f}", ha="center", fontsize=9)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("mAP50 Improvement")
axes[1].set_title("Per-Iteration Improvement", fontweight="bold")
axes[1].set_xticks(iter_nums)
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# Panel 3: Cumulative records and cost
cum_records = np.cumsum(records_per_iter)
cum_cost = np.cumsum(costs_per_iter)
ax3 = axes[2]
ax3_twin = ax3.twinx()
ax3.bar(iter_nums, records_per_iter, color="#3498db", alpha=0.7, edgecolor="white", label="Records")
ax3_twin.plot(iter_nums, cum_cost, "s-", color="#e74c3c", lw=2, markersize=6, label="Cumulative cost")
ax3.set_xlabel("Iteration")
ax3.set_ylabel("Records Generated", color="#3498db")
ax3_twin.set_ylabel("Cumulative Cost ($)", color="#e74c3c")
ax3.set_title("Generation Volume & Cost", fontweight="bold")
ax3.set_xticks(iter_nums)
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

plt.tight_layout()
plt.show()

# ── Delta from baseline ──────────────────────────────────────────────
delta = al_result.final_map50 - baseline_result.best_map50
print(f"Baseline mAP50:      {baseline_result.best_map50:.4f}")
print(f"Final AL mAP50:      {al_result.final_map50:.4f}")
print(f"Improvement:         {'+' if delta >= 0 else ''}{delta:.4f}")
print(f"Total records:       {int(cum_records[-1])} synthetic images")
print(f"Total cost:          ${al_result.total_cost_usd:.2f}")

---
## Step 7: Per-Bucket & Per-Region Recall Evolution

The `ModelPerformanceProfile` breaks down recall by **size bucket** (tiny/small/medium/large) and **spatial region** (3x3 grid). This is the core feedback signal: buckets/regions with recall < 0.5 get boosted priority in the next iteration's synthesis strategy.

Watch how recall evolves across iterations — ideally, weak areas improve as the loop generates targeted data.

In [ ]:
# ── Per-bucket recall across iterations ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel 1: Bucket recall per iteration (grouped bars)
n_iters = len(iters)
x = np.arange(len(BUCKET_ORDER))
width = 0.8 / max(n_iters, 1)

for j, ir in enumerate(iters):
    recalls = [ir.profile.per_bucket_map.get(BBoxSizeBucket(b), 0.0) for b in BUCKET_ORDER]
    offset = (j - n_iters / 2 + 0.5) * width
    bars = axes[0].bar(x + offset, recalls, width, label=f"Iter {ir.iteration + 1}",
                       color=plt.cm.viridis(j / max(n_iters - 1, 1)), edgecolor="white")
    for xi, v in zip(x + offset, recalls):
        if v > 0:
            axes[0].text(xi, v + 0.02, f"{v:.2f}", ha="center", fontsize=7, rotation=0)

axes[0].axhline(0.5, color="#e74c3c", ls="--", lw=1, alpha=0.7, label="Weakness threshold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(BUCKET_ORDER)
axes[0].set_ylabel("Recall")
axes[0].set_ylim(0, 1.15)
axes[0].set_title("Per-Bucket Recall Across Iterations", fontweight="bold")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3, axis="y")

# Panel 2: False negatives by bucket (stacked area)
for j, ir in enumerate(iters):
    fn_counts = [ir.profile.false_negative_buckets.get(BBoxSizeBucket(b), 0) for b in BUCKET_ORDER]
    offset = (j - n_iters / 2 + 0.5) * width
    axes[1].bar(x + offset, fn_counts, width, label=f"Iter {ir.iteration + 1}",
                color=plt.cm.viridis(j / max(n_iters - 1, 1)), edgecolor="white")

axes[1].set_xticks(x)
axes[1].set_xticklabels(BUCKET_ORDER)
axes[1].set_ylabel("False Negatives")
axes[1].set_title("False Negatives by Bucket (should decrease)", fontweight="bold")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

# ── Per-region recall heatmaps ───────────────────────────────────────
fig, axes = plt.subplots(1, n_iters, figsize=(5 * n_iters, 4))
if n_iters == 1:
    axes = [axes]

# Shared color scale across all iterations
all_recalls = []
for ir in iters:
    for reg in REGION_ORDER:
        all_recalls.append(ir.profile.per_region_map.get(SpatialRegion(reg), 0.0))
vmin, vmax = 0.0, max(all_recalls) if all_recalls else 1.0

for j, (ax, ir) in enumerate(zip(axes, iters)):
    grid = np.zeros((3, 3))
    for i, reg in enumerate(REGION_ORDER):
        r, c = divmod(i, 3)
        grid[r, c] = ir.profile.per_region_map.get(SpatialRegion(reg), 0.0)

    im = ax.imshow(grid, cmap="RdYlGn", vmin=vmin, vmax=max(vmax, 0.01))
    for r in range(3):
        for c in range(3):
            color = "white" if grid[r, c] < 0.5 else "black"
            ax.text(c, r, f"{grid[r, c]:.2f}", ha="center", va="center",
                    fontsize=11, fontweight="bold", color=color)
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["Left", "Center", "Right"])
    ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["Top", "Middle", "Bottom"])
    ax.set_title(f"Iteration {ir.iteration + 1} (mAP50={ir.map50:.3f})", fontweight="bold")

fig.colorbar(im, ax=axes, shrink=0.8, label="Recall")
fig.suptitle("Per-Region Recall Evolution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Step 8: Exploration vs Exploitation

In the first iteration, all generation tasks are **exploration** (filling dataset gaps). In later iterations, tasks targeting model-weak areas get a **+0.1 priority boost** (exploitation). This section shows how the strategy shifts across iterations.

The `ModelPerformanceProfile` identifies weak buckets (recall < 0.5) which then influence the next iteration's task priorities.

In [ ]:
# ── Show weak buckets/regions identified at each iteration ────────────
for ir in iters:
    print(f"\n--- After Iteration {ir.iteration + 1} (mAP50={ir.map50:.3f}) ---")

    weak_buckets = {b.value: r for b, r in ir.profile.per_bucket_map.items() if r < 0.5}
    weak_regions = {r.value: v for r, v in ir.profile.per_region_map.items() if v < 0.5}

    if weak_buckets:
        print(f"  Weak buckets (recall < 0.5):  {', '.join(f'{b}={r:.2f}' for b, r in weak_buckets.items())}")
    else:
        print(f"  Weak buckets: none (all >= 0.5)")

    if weak_regions:
        print(f"  Weak regions (recall < 0.5):  {', '.join(f'{r}={v:.2f}' for r, v in weak_regions.items())}")
    else:
        print(f"  Weak regions: none (all >= 0.5)")

    # Show per-class recall
    if ir.profile.per_class_map:
        print(f"  Per-class mAP50:")
        for cls_id, val in sorted(ir.profile.per_class_map.items()):
            name = dataset.class_names[cls_id] if cls_id < len(dataset.class_names) else f"class_{cls_id}"
            print(f"    {name}: {val:.3f}")

# ── Priority distribution per iteration ───────────────────────────────
# Re-generate strategies with model profiles from each iteration
# to show how priorities shift
print("\n" + "=" * 60)
print("Strategy Priority Summary")
print("=" * 60)
print(f"  Iteration 0 (initial):  Pure exploration, no model feedback")

for ir in iters:
    # Count how many tasks would get exploitation boost
    weak_b = {b for b, r in ir.profile.per_bucket_map.items() if r < 0.5}
    weak_r = {r for r, v in ir.profile.per_region_map.items() if v < 0.5}
    boosted = 0
    for task in initial_strategy.generation_tasks:
        overlap = (
            any(b in weak_b for b in task.target_size_buckets)
            or any(r in weak_r for r in task.target_regions)
        )
        if overlap:
            boosted += 1
    total_tasks = len(initial_strategy.generation_tasks)
    print(f"  After Iter {ir.iteration + 1}: {boosted}/{total_tasks} tasks would get exploitation boost (+0.1 priority)")

---
## Step 9: Baseline vs Final Model — Failure Case Comparison

Find validation images where the **baseline model** misses the most ground-truth detections, then compare against the **final active learning model** on the same images.

Three columns per image:
- **Ground Truth** — human annotations
- **Baseline** — model trained on original data only
- **Final AL** — model after N iterations of active learning

In [ ]:
from ultralytics import YOLO
from synthdet.types import BBox, AnnotationSource

# ── Load both models ──────────────────────────────────────────────────
baseline_model = YOLO(baseline_result.best_weights)
al_model = YOLO(str(al_result.final_weights))

CONF_THRESH = 0.25
IOU_THRESH = 0.5


def _iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0.0


def _to_xyxy(bbox, w, h):
    return (
        (bbox.x_center - bbox.width / 2) * w,
        (bbox.y_center - bbox.height / 2) * h,
        (bbox.x_center + bbox.width / 2) * w,
        (bbox.y_center + bbox.height / 2) * h,
    )


def _count_misses(gt_xyxy, pred_xyxy, iou_thresh=0.5):
    matched = set()
    hits = 0
    for gt in gt_xyxy:
        best_iou, best_pi = 0, -1
        for pi, pred in enumerate(pred_xyxy):
            if pi in matched:
                continue
            iou = _iou(gt, pred)
            if iou > best_iou:
                best_iou = iou
                best_pi = pi
        if best_iou >= iou_thresh and best_pi >= 0:
            matched.add(best_pi)
            hits += 1
    return len(gt_xyxy) - hits


def _results_to_bboxes(results, w, h):
    out = []
    if results and results[0].boxes is not None and len(results[0].boxes):
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().tolist()
            cls_id = int(box.cls[0].cpu().item())
            conf = float(box.conf[0].cpu().item())
            out.append((
                BBox(
                    class_id=cls_id,
                    x_center=((x1 + x2) / 2) / w,
                    y_center=((y1 + y2) / 2) / h,
                    width=(x2 - x1) / w,
                    height=(y2 - y1) / h,
                    source=AnnotationSource.UNKNOWN,
                ),
                conf,
            ))
    return out


# ── Score validation images by baseline misses ────────────────────────
val_records = dataset.valid
failure_scores = []

for idx, rec in enumerate(val_records):
    if not rec.bboxes:
        continue
    img = rec.load_image()
    h, w = img.shape[:2]
    gt_xyxy = [_to_xyxy(b, w, h) for b in rec.bboxes]
    results = baseline_model.predict(img, imgsz=MAX_IMAGE_SIZE, conf=CONF_THRESH, verbose=False)
    preds = _results_to_bboxes(results, w, h)
    pred_xyxy = [_to_xyxy(b, w, h) for b, _ in preds]
    missed = _count_misses(gt_xyxy, pred_xyxy, IOU_THRESH)
    failure_scores.append((idx, missed, len(rec.bboxes)))
    rec.image = None

failure_scores.sort(key=lambda x: (-x[1], -x[2]))
top_failures = failure_scores[:4]

print("Validation images where baseline fails most:")
for idx, missed, total in top_failures:
    print(f"  val[{idx}]: {missed}/{total} ground truths missed")

In [ ]:
# ── Side-by-side: Ground Truth | Baseline | Final AL Model ────────────
n_imgs = len(top_failures)
fig, axes = plt.subplots(n_imgs, 3, figsize=(18, 5 * n_imgs))
if n_imgs == 1:
    axes = axes[np.newaxis, :]

for row, (idx, baseline_missed, total_gt) in enumerate(top_failures):
    rec = val_records[idx]
    img = resize_if_needed(rec.load_image(), MAX_IMAGE_SIZE)
    img_rgb = to_rgb(img)
    h, w = img.shape[:2]
    gt_xyxy = [_to_xyxy(b, w, h) for b in rec.bboxes]

    # Column 1: Ground Truth
    show_bboxes(img_rgb, rec.bboxes, dataset.class_names,
                ax=axes[row, 0], title=f"Ground Truth ({total_gt} boxes)")

    # Column 2: Baseline predictions
    base_results = baseline_model.predict(img, imgsz=MAX_IMAGE_SIZE, conf=CONF_THRESH, verbose=False)
    base_preds = _results_to_bboxes(base_results, w, h)
    show_preds(img_rgb, base_preds, dataset.class_names, ax=axes[row, 1],
               title=f"Baseline ({len(base_preds)} preds, {baseline_missed} missed)")

    # Column 3: Active Learning model predictions
    al_results = al_model.predict(img, imgsz=MAX_IMAGE_SIZE, conf=CONF_THRESH, verbose=False)
    al_preds = _results_to_bboxes(al_results, w, h)
    al_pred_xyxy = [_to_xyxy(b, w, h) for b, _ in al_preds]
    al_missed = _count_misses(gt_xyxy, al_pred_xyxy, IOU_THRESH)
    show_preds(img_rgb, al_preds, dataset.class_names, ax=axes[row, 2],
               title=f"Final AL ({len(al_preds)} preds, {al_missed} missed)")

    rec.image = None

for col, label in enumerate(["Ground Truth", "Baseline Model", f"After {len(iters)} AL Iterations"]):
    axes[0, col].text(0.5, 1.12, label, transform=axes[0, col].transAxes,
                      ha="center", fontsize=12, fontweight="bold")

fig.suptitle("Baseline vs Active Learning Model on Hardest Validation Images",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Summary ──────────────────────────────────────────────────────────
total_baseline_missed = sum(m for _, m, _ in top_failures)
total_gt = sum(t for _, _, t in top_failures)
total_al_missed = 0
for idx, _, _ in top_failures:
    rec = val_records[idx]
    img = rec.load_image()
    h, w = img.shape[:2]
    gt_xyxy = [_to_xyxy(b, w, h) for b in rec.bboxes]
    al_results = al_model.predict(img, imgsz=MAX_IMAGE_SIZE, conf=CONF_THRESH, verbose=False)
    al_preds = _results_to_bboxes(al_results, w, h)
    al_pred_xyxy = [_to_xyxy(b, w, h) for b, _ in al_preds]
    total_al_missed += _count_misses(gt_xyxy, al_pred_xyxy, IOU_THRESH)
    rec.image = None

print(f"\nOn the 4 hardest images ({total_gt} ground truth boxes):")
print(f"  Baseline missed:     {total_baseline_missed}/{total_gt}")
print(f"  Active Learning:     {total_al_missed}/{total_gt}")
improvement = total_baseline_missed - total_al_missed
if improvement > 0:
    print(f"  AL recovered {improvement} detection(s) the baseline missed")
elif improvement == 0:
    print(f"  Both models missed the same detections")
else:
    print(f"  Baseline performed better by {-improvement} detection(s)")

---
## Step 10: Final Dashboard

Summary of the entire active learning run: dataset growth, performance improvement, cost, and training time.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Dataset growth per iteration
cum_records = np.cumsum(records_per_iter)
total_with_orig = [len(dataset.train) + int(c) for c in cum_records]
axes[0].bar(iter_nums, total_with_orig, color="#2ecc71", edgecolor="white", label="Total (orig + synth)")
axes[0].axhline(len(dataset.train), color="#95a5a6", ls="--", lw=2, label=f"Original ({len(dataset.train)})")
for x, y in zip(iter_nums, total_with_orig):
    axes[0].text(x, y + 5, str(y), ha="center", fontsize=9)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Training Images")
axes[0].set_title("Cumulative Training Set Size", fontweight="bold")
axes[0].set_xticks(iter_nums)
axes[0].legend(fontsize=8)

# Panel 2: Baseline vs AL metrics
metric_labels = ["mAP50", "mAP50-95"]
baseline_vals = [baseline_result.best_map50, baseline_result.best_map50_95]
al_vals = [al_result.final_map50, al_result.iterations[-1].profile.overall_map50_95]
x = np.arange(len(metric_labels))
w = 0.3
axes[1].bar(x - w/2, baseline_vals, w, label="Baseline", color="#95a5a6", edgecolor="white")
axes[1].bar(x + w/2, al_vals, w, label=f"After {len(iters)} AL Iters", color="#2ecc71", edgecolor="white")
for i, (bv, av) in enumerate(zip(baseline_vals, al_vals)):
    axes[1].text(i - w/2, bv + 0.01, f"{bv:.3f}", ha="center", fontsize=9)
    axes[1].text(i + w/2, av + 0.01, f"{av:.3f}", ha="center", fontsize=9)
    delta = av - bv
    sign = "+" if delta >= 0 else ""
    axes[1].annotate(
        f"{sign}{delta:.3f}", xy=(i + w/2, av + 0.04),
        fontsize=10, fontweight="bold", ha="center",
        color="#27ae60" if delta > 0 else "#e74c3c",
    )
axes[1].set_xticks(x)
axes[1].set_xticklabels(metric_labels)
axes[1].set_ylim(0, 1)
axes[1].set_title("Detection Metrics: Baseline vs AL", fontweight="bold")
axes[1].legend()

# Panel 3: Summary text
delta_50 = al_result.final_map50 - baseline_result.best_map50
total_synth = int(cum_records[-1])
info_text = (
    f"Active Learning Summary\n"
    f"{'=' * 30}\n"
    f"  Iterations:       {len(iters)} ({al_result.stopped_reason})\n"
    f"  Synthetic images: {total_synth}\n"
    f"  API cost:         ${al_result.total_cost_usd:.2f}\n"
    f"  Training time:    {al_result.total_training_time_seconds:.0f}s\n"
    f"\n"
    f"  Baseline mAP50:   {baseline_result.best_map50:.4f}\n"
    f"  Final mAP50:      {al_result.final_map50:.4f}\n"
    f"  Improvement:      {'+' if delta_50 >= 0 else ''}{delta_50:.4f}\n"
)
axes[2].text(0.05, 0.5, info_text, transform=axes[2].transAxes,
             fontsize=10, verticalalignment="center", fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.5", facecolor="#ecf0f1", edgecolor="#bdc3c7"))
axes[2].axis("off")
axes[2].set_title("Summary", fontweight="bold")

plt.tight_layout()
plt.show()